# REACTO — constrained multi-objective Bayesian optimisation

A condensed, self-contained reference implementation of the algorithm used to
optimise the **MD_DMIPP phosphorylation** campaign. It reproduces, step by
step, what REACTO does between one experiment and the next.

REACTO is a Bayesian optimisation framework for chemical reactions built on
[BoFire](https://github.com/experimental-design/bofire) and
[BoTorch](https://botorch.org). A campaign is a closed loop:

1. the reaction space and the objectives are declared once, as a *domain*;
2. a space-filling **initial design** is drawn inside that space;
3. the experiments performed so far are used to fit one Gaussian-process
   surrogate **per objective** and one **per outcome constraint**;
4. a constrained **qLogNEHVI** acquisition function is maximised over the
   reaction space, giving the next set of conditions to run;
5. the chemist runs it, records the outcomes, and the loop returns to step 3.

Nothing is batched and nothing is simulated: one experiment is proposed, one
experiment is performed, the models are refitted.

## The optimisation problem

| | |
|---|---|
| Reaction | MD_DMIPP phosphorylation |
| Parameters | `DMIPP` (1.0–6.7 equiv), `BUOH` (1.0–10.9 equiv), `T` (25–80 °C), `RES` (0.5–2.5 min, five levels) |
| Objectives | maximise `YIELD`, maximise `STY` |
| Outcome constraint | `YIELD` ≥ 0.60 |
| Process constraint | `DMIPP` ≤ `BUOH` |

Two competing objectives make this a Pareto problem: there is no single best
experiment, but a front of non-dominated trade-offs between conversion and
throughput. The outcome constraint restricts that front to the region a
chemist would accept.

## 0. Setup

Python 3.11, `bofire==0.3.1`, `botorch==0.17.0`, `gpytorch==1.15.2`,
`torch==2.10.0`. All computations are in double precision, which BoTorch's
acquisition optimisers require for numerical stability.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import torch

import bofire.strategies.api as strategies
from bofire.data_models.api import Domain, Inputs, Outputs, Constraints
from bofire.data_models.features.api import (
    ContinuousInput, DiscreteInput, CategoricalInput, CategoricalDescriptorInput,
    ContinuousOutput,
)
from bofire.data_models.objectives.api import MaximizeObjective, MinimizeObjective
from bofire.data_models.constraints.api import LinearInequalityConstraint
from bofire.data_models.strategies.api import RandomStrategy
from bofire.data_models.enum import SamplingMethodEnum

from botorch import fit_gpytorch_mll
from botorch.exceptions import BadInitialCandidatesWarning
from botorch.models.gp_regression import SingleTaskGP
from botorch.models.model_list_gp_regression import ModelListGP
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.acquisition.multi_objective.logei import (
    qLogNoisyExpectedHypervolumeImprovement,
)
from botorch.acquisition.multi_objective.objective import IdentityMCMultiOutputObjective
from botorch.optim.optimize import optimize_acqf, optimize_acqf_mixed
from botorch.utils.multi_objective.hypervolume import Hypervolume
from botorch.utils.multi_objective.pareto import is_non_dominated
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood

warnings.filterwarnings("ignore", category=BadInitialCandidatesWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

TKWARGS = {
    "dtype": torch.double,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

# Acquisition-optimiser settings: REACTO's defaults, and the only numerical
# knobs of the whole procedure.
BATCH_SIZE   = 1      # one experiment proposed at a time
NUM_RESTARTS = 10     # multi-start maximisation of the acquisition function
RAW_SAMPLES  = 128    # raw candidates used to initialise those restarts
MC_SAMPLES   = 128    # quasi-Monte-Carlo samples of the GP posterior
SEED         = 0      # set to None for an unseeded (non-reproducible) run

## 1. The reaction space

The domain is the single declarative object that carries the whole problem:
what can be varied and within which bounds, what is being optimised and in
which direction, and which combinations of parameters are forbidden.

Three parameter types are supported. **Continuous** inputs are box-bounded real
variables; **discrete** inputs take values from an explicit list, which is how a
residence time restricted to the pump settings actually available is expressed;
**categorical** inputs enumerate unordered alternatives such as solvents,
ligands or bases, and may carry physico-chemical descriptors so that the
surrogate interpolates between them rather than treating them as unrelated
labels. This campaign uses the first two.

Each objective declares a direction and the bounds of its useful range. Those
bounds are not cosmetic: they supply the **reference point** of the hypervolume
computation in §4, i.e. the worst outcome still considered informative.

`DMIPP ≤ BUOH` is a *process* constraint, known before any experiment is run,
and is written as a linear inequality `1·DMIPP + (−1)·BUOH ≤ 0`. It is enforced
exactly, both when the initial design is drawn and when the acquisition
function is maximised — such conditions are never proposed.

`YIELD ≥ 0.60` is an *outcome* constraint of a different nature: whether it
holds at a given set of conditions is unknown until the experiment has been
run, so it cannot be imposed geometrically. It is learned instead, as §4
describes.

In [ ]:
domain = Domain(
    inputs=Inputs(features=[
        ContinuousInput(key="DMIPP", bounds=[1.0, 6.7],   unit="equiv"),
        ContinuousInput(key="BUOH",  bounds=[1.0, 10.9],  unit="equiv"),
        ContinuousInput(key="T",     bounds=[25.0, 80.0], unit="degC"),
        DiscreteInput(key="RES", values=[0.5, 1.0, 1.5, 2.0, 2.5], unit="min"),
    ]),
    outputs=Outputs(features=[
        ContinuousOutput(key="YIELD", objective=MaximizeObjective(w=1.0, bounds=(0.0, 1.0))),
        ContinuousOutput(key="STY",   objective=MaximizeObjective(w=1.0, bounds=(0.0, 73.2))),
    ]),
    constraints=Constraints(constraints=[
        # DMIPP <= BUOH  ->  1*DMIPP + (-1)*BUOH <= 0
        LinearInequalityConstraint(
            features=["DMIPP", "BUOH"], coefficients=[1.0, -1.0], rhs=0.0,
        ),
    ]),
)

# The outcome constraint is carried separately: it is handled by the surrogate,
# not by the geometry of the domain.
OUTCOME_CONSTRAINT = {"objective": "YIELD", "direction": ">=", "threshold": 0.60}

print(domain.inputs.get_keys(), "->", domain.outputs.get_keys())

## 2. Initial design

Bayesian optimisation needs something to learn from before it can propose
anything. REACTO draws that first set of conditions with a space-filling
design over the declared space — Latin hypercube by default, with random,
Sobol and constrained *k*-means also available — subject to the linear
constraints, so that the surrogates start from conditions spread through the
reaction space rather than clustered around a chemist's prior.

Passing a seed matters more than it appears. The sampler carries its own
random generator, which neither `np.random.seed` nor `torch.manual_seed`
reaches; without an explicit seed a design cannot be replayed.

In [ ]:
def initial_design(domain, n_points, method="LHS", seed=None):
    # Draw a space-filling design honouring the domain's linear constraints.
    # method: 'LHS', 'UNIFORM' or 'SOBOL'.
    data_model = RandomStrategy(
        domain=domain,
        fallback_sampling_method=SamplingMethodEnum[method],
        **({} if seed is None else {"seed": seed}),
    )
    return strategies.map(data_model).ask(n_points)


# The conditions that opened this campaign were generated this way; they are
# reproduced verbatim in §6 together with their measured outcomes.
initial_design(domain, 5, method="LHS", seed=SEED).round(2)

## 3. Encoding the reaction space

BoTorch optimises over the unit cube, so the domain has to be mapped onto
$[0,1]^D$ and the proposal mapped back into reaction conditions.

* **continuous** → min–max scaled onto $[0,1]$;
* **discrete** → scaled the same way over its range, relaxed during the search
  and snapped back to the nearest declared level on decoding;
* **categorical** → one-hot over its $k$ categories, occupying $k$ columns;
  the search enumerates the categories explicitly (§5) rather than relaxing
  them, so the one-hot block always decodes to exactly one category.

The linear constraints must travel with this change of variables. Substituting
$v_i = l_i + x_i (u_i - l_i)$ into $\sum_i a_i v_i \le r$ gives
$\sum_i a_i (u_i - l_i)\, x_i \le r - \sum_i a_i l_i$, which is then negated
into the $\sum_i c_i x_i \ge r'$ form BoTorch expects.

In [ ]:
def build_encoding(domain):
    # Column layout of the encoded search space [0,1]^D.
    col_info, start = [], 0
    for feat in domain.inputs.features:
        if isinstance(feat, ContinuousInput):
            ci = {"key": feat.key, "type": "continuous",
                  "bounds": tuple(feat.bounds), "width": 1}
        elif isinstance(feat, DiscreteInput):
            values = sorted(feat.values)
            ci = {"key": feat.key, "type": "discrete", "values": values,
                  "bounds": (values[0], values[-1]), "width": 1}
        elif isinstance(feat, (CategoricalInput, CategoricalDescriptorInput)):
            categories = list(feat.categories)
            ci = {"key": feat.key, "type": "categorical",
                  "categories": categories, "width": len(categories)}
        else:
            raise TypeError(f"unsupported feature type: {type(feat).__name__}")
        ci["start"] = start
        start += ci["width"]
        col_info.append(ci)
    return col_info, start


def encode_experiments(experiments, col_info):
    # Experiments in original units -> (n, D) tensor in [0,1]^D.
    rows = []
    for _, row in experiments.iterrows():
        encoded = []
        for ci in col_info:
            if ci["type"] in ("continuous", "discrete"):
                lb, ub = ci["bounds"]
                x = (float(row[ci["key"]]) - lb) / (ub - lb) if ub > lb else 0.5
                encoded.append(min(max(x, 0.0), 1.0))
            else:
                encoded.extend(1.0 if c == str(row[ci["key"]]) else 0.0
                               for c in ci["categories"])
        rows.append(encoded)
    return torch.tensor(rows, **TKWARGS)


def decode_candidate(x, col_info):
    # A single point of [0,1]^D -> a dict of reaction conditions.
    conditions = {}
    for ci in col_info:
        i, n = ci["start"], ci["width"]
        if ci["type"] == "continuous":
            lb, ub = ci["bounds"]
            conditions[ci["key"]] = lb + float(x[i]) * (ub - lb)
        elif ci["type"] == "discrete":
            lb, ub = ci["bounds"]
            relaxed = lb + float(x[i]) * (ub - lb)
            conditions[ci["key"]] = min(ci["values"], key=lambda v: abs(v - relaxed))
        else:
            conditions[ci["key"]] = ci["categories"][int(x[i:i + n].argmax())]
    return conditions

In [ ]:
def encode_linear_constraints(domain, col_info):
    # BoFire linear inequalities -> BoTorch (indices, coefficients, rhs) triples.
    # BoFire states them as sum(a_i * v_i) <= r in original units; BoTorch expects
    # sum(c_j * x_j) >= r' in the encoded space. Both the change of variables and
    # the change of sign are applied here.
    by_key, encoded = {ci["key"]: ci for ci in col_info}, []
    if domain.constraints is None:
        return encoded
    for con in domain.constraints.constraints:
        if not isinstance(con, LinearInequalityConstraint):
            continue   # categorical exclusions are handled by the enumeration of §5
        indices, coefficients, shift = [], [], 0.0
        for key, a in zip(con.features, con.coefficients):
            ci = by_key[key]
            lb, ub = ci["bounds"]
            indices.append(ci["start"])
            coefficients.append(-a * (ub - lb))    # sign flipped for the '>=' form
            shift += a * lb
        encoded.append((
            torch.tensor(indices, dtype=torch.long, device=TKWARGS["device"]),
            torch.tensor(coefficients, **TKWARGS),
            -(float(con.rhs) - shift),
        ))
    return encoded


def build_fixed_features(col_info):
    # One entry per combination of categories, or None if there are none. Each
    # entry pins the one-hot block of every categorical parameter, so the
    # acquisition function is maximised over the continuous parameters once per
    # combination and the best of those maxima is returned.
    blocks = [ci for ci in col_info if ci["type"] == "categorical"]
    if not blocks:
        return None
    from itertools import product
    fixed_list = []
    for combo in product(*[range(ci["width"]) for ci in blocks]):
        fixed = {}
        for ci, chosen in zip(blocks, combo):
            for k in range(ci["width"]):
                fixed[ci["start"] + k] = 1.0 if k == chosen else 0.0
        fixed_list.append(fixed)
    return fixed_list


col_info, n_dim = build_encoding(domain)
print(f"encoded dimension D = {n_dim}")
encode_linear_constraints(domain, col_info)

## 4. Surrogate models

One independent `SingleTaskGP` is fitted per output — a Matérn-5/2 kernel with
automatic relevance determination and a fitted noise term — and the lot is
collected into a `ModelListGP`, whose marginal log-likelihoods are summed and
maximised jointly:

$$\mathrm{ModelListGP} = \big[\; \mathrm{GP}_{\mathrm{YIELD}},\; \mathrm{GP}_{\mathrm{STY}},\; \mathrm{GP}_{c}\;\big]$$

Two conventions are applied on the way in.

**Direction.** BoTorch maximises, so an objective declared for minimisation has
its sign flipped. Both objectives are maximised here, so nothing is flipped.

**The outcome constraint becomes a third GP.** Rather than filtering
suggestions after the fact, the constraint is modelled as an extra output

$$c(\mathbf{x}) = \tau - \mathrm{YIELD}(\mathbf{x}) \qquad
  (\text{or } \mathrm{YIELD}(\mathbf{x}) - \tau \text{ for a } \le \text{ constraint})$$

with threshold $\tau$ and the BoTorch convention that **$c(\mathbf{x}) \le 0$
means feasible**. The surrogate therefore carries a posterior belief about
*where* the constraint holds, and the acquisition function of §5 weights each
candidate by the probability that it does. An experiment that violates the
threshold is not wasted: it teaches the third GP where the infeasible region
lies.

The **reference point** of the hypervolume is read off the objective bounds
declared in §1 — the lower bound for a maximised objective, minus the upper
bound for a minimised one. Rescaling an objective rescales the hypervolume by
the same factor and leaves the arg-max, hence the proposal, unchanged.

In [ ]:
def fit_surrogates(domain, experiments, outcome_constraint, col_info):
    # Fit [one GP per objective] + [one GP for the outcome constraint].
    objective_feats = domain.outputs.features

    train_x = encode_experiments(experiments, col_info)

    # Objectives, sign-flipped where the declared direction is minimisation.
    train_obj = torch.tensor([
        [(-1.0 if isinstance(f.objective, MinimizeObjective) else 1.0) * float(row[f.key])
         for f in objective_feats]
        for _, row in experiments.iterrows()
    ], **TKWARGS)

    # Outcome constraint, in the convention c(x) <= 0 <=> feasible.
    key       = outcome_constraint["objective"]
    threshold = float(outcome_constraint["threshold"])
    sign      = 1.0 if outcome_constraint["direction"] == ">=" else -1.0
    train_con = torch.tensor([[sign * (threshold - float(row[key]))]
                              for _, row in experiments.iterrows()], **TKWARGS)

    # Reference point, from the objective bounds declared with the domain.
    ref_point = [
        -float(f.objective.upper_bound) if isinstance(f.objective, MinimizeObjective)
        else float(f.objective.lower_bound)
        for f in objective_feats
    ]

    train_y = torch.cat([train_obj, train_con], dim=-1)      # (n, n_obj + 1)
    model = ModelListGP(*[
        SingleTaskGP(train_x, train_y[..., i:i + 1]) for i in range(train_y.shape[-1])
    ])
    fit_gpytorch_mll(SumMarginalLogLikelihood(model.likelihood, model))

    return model, train_x, train_obj, train_con, ref_point

## 5. Acquisition function and proposal

The next experiment is the maximiser of the **constrained log noisy expected
hypervolume improvement**, qLogNEHVI. Expected hypervolume improvement scores a
candidate by how much it is expected to push out the Pareto front, measured as
the volume it adds relative to the reference point; the *noisy* variant treats
the observed outcomes as noisy realisations rather than ground truth, which is
the honest assumption for a bench measurement; the *log* formulation is the
numerically stable reparametrisation that keeps gradients informative where
improvement probabilities are vanishingly small.

The expectation is taken over quasi-Monte-Carlo draws from the joint GP
posterior. `constraints=[lambda Z: Z[..., -1]]` points the acquisition function
at the last output of the `ModelListGP` — the constraint GP — so that each
posterior draw is weighted by the probability that $c(\mathbf{x}) \le 0$ there.
`prune_baseline=True` discards baseline points that cannot contribute to the
improvement, which keeps the cost manageable as the campaign grows.

The maximisation itself is a multi-start L-BFGS-B run over the encoded space,
subject to the linear constraints of §3. With categorical parameters present,
`optimize_acqf_mixed` runs that maximisation once per combination of
categories and keeps the best; without them, a single `optimize_acqf` call
suffices. The winner is decoded back into reaction conditions.

*Single-objective campaigns.* With one objective and no outcome constraint the
same machinery reduces to a single GP under **qLogNEI** — noisy expected
improvement over the incumbent best, instead of over the Pareto front.

In [ ]:
def suggest_next_experiment(domain, experiments, outcome_constraint,
                            q=BATCH_SIZE, seed=None, return_model=False):
    # Propose the next q experiments by maximising constrained qLogNEHVI.
    col_info, n_dim = build_encoding(domain)
    model, train_x, train_obj, train_con, ref_point = fit_surrogates(
        domain, experiments, outcome_constraint, col_info
    )

    acquisition = qLogNoisyExpectedHypervolumeImprovement(
        model=model,
        ref_point=ref_point,
        X_baseline=train_x,
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([MC_SAMPLES]),
                                      **({} if seed is None else {"seed": seed})),
        prune_baseline=True,
        # which outputs of the ModelListGP are the objectives ...
        objective=IdentityMCMultiOutputObjective(outcomes=list(range(train_obj.shape[-1]))),
        # ... and which one is the constraint: the last, feasible where <= 0
        constraints=[lambda Z: Z[..., -1]],
    )

    bounds = torch.zeros(2, n_dim, **TKWARGS)
    bounds[1] = 1.0
    shared = dict(
        acq_function=acquisition, bounds=bounds, q=q,
        num_restarts=NUM_RESTARTS, raw_samples=RAW_SAMPLES,
        inequality_constraints=encode_linear_constraints(domain, col_info) or None,
        options={"batch_limit": 5, "maxiter": 200},
    )

    fixed_features_list = build_fixed_features(col_info)
    if fixed_features_list:
        candidates, _ = optimize_acqf_mixed(fixed_features_list=fixed_features_list, **shared)
    else:
        candidates, _ = optimize_acqf(**shared)

    proposal = pd.DataFrame([decode_candidate(x, col_info) for x in candidates.detach()])
    return (proposal, model, col_info) if return_model else proposal

## 6. The campaign

The 18 experiments that opened the campaign: the initial design of §2, with the
outcomes measured for each. `YIELD` is a fraction and `STY` is in g L⁻¹ h⁻¹.

In [ ]:
EXPERIMENTS = pd.DataFrame([
    #  DMIPP  BUOH     T   RES   YIELD   STY
    (   5.0,   8.2,  40.0,  2.5,  0.27,   2.9),
    (   2.2,   9.0,  65.0,  1.0,  0.26,   3.1),
    (   2.2,   6.2,  70.0,  2.5,  0.54,   2.6),
    (   5.0,   8.5,  35.0,  1.0,  0.06,   0.1),
    (   5.0,   8.2,  65.0,  0.5,  0.24,  13.0),
    (   2.4,   6.3,  35.0,  0.5,  0.04,   1.1),
    (   2.2,   4.4,  40.0,  2.0,  0.35,   2.1),
    (   5.0,   8.2,  70.0,  2.0,  0.43,   5.9),
    (   2.2,   8.8,  40.0,  2.0,  0.09,   0.5),
    (   2.2,   4.2,  65.0,  1.0,  0.59,   7.2),
    (   3.2,   3.2,  70.0,  1.0,  0.44,   7.8),
    (   3.2,   4.7,  60.0,  1.5,  0.55,   6.4),
    (   1.2,   4.5,  65.0,  1.0,  0.44,   2.2),
    (   2.6,   4.5,  65.0,  1.5,  0.53,   5.0),
    (   4.6,   4.6,  60.0,  2.5,  0.48,   4.8),
    (   3.4,   5.5,  75.0,  1.5,  0.44,   5.4),
    (   2.6,   2.6,  60.0,  1.0,  0.54,   7.7),
    (   2.6,   3.6,  60.0,  2.0,  0.61,   4.4),
], columns=["DMIPP", "BUOH", "T", "RES", "YIELD", "STY"])

print(f"{len(EXPERIMENTS)} experiments, "
      f"{int((EXPERIMENTS['YIELD'] >= OUTCOME_CONSTRAINT['threshold']).sum())} of them feasible")
EXPERIMENTS.head()

In [ ]:
proposal, model, col_info = suggest_next_experiment(
    domain, EXPERIMENTS, OUTCOME_CONSTRAINT, q=1, seed=SEED, return_model=True,
)
proposal.round(2)

### What the surrogates say about the proposal

Querying the three GPs at the proposed conditions separates the two things the
acquisition function trades off: the predicted outcomes, and how confident the
models are that the constraint holds there. Under the Gaussian posterior of the
constraint GP, the probability of feasibility is
$P(c(\mathbf{x}) \le 0) = \Phi(-\mu_c/\sigma_c)$.

A candidate with a high posterior mean but a wide posterior is the model
proposing an *exploratory* experiment; a narrow posterior near the current
front is *exploitation*. qLogNEHVI arbitrates between the two without either
being specified by hand.

In [ ]:
def inspect_proposal(model, proposal, domain, outcome_constraint, col_info):
    x = encode_experiments(proposal, col_info)
    model.eval()
    with torch.no_grad():
        posterior = model.posterior(x)
        n_out = len(domain.outputs.features) + 1
        mean  = posterior.mean.reshape(-1, n_out)
        sigma = posterior.variance.sqrt().reshape(-1, n_out)

    table = pd.DataFrame(
        {"posterior mean": mean[0].cpu().numpy(),
         "posterior sd":   sigma[0].cpu().numpy()},
        index=domain.outputs.get_keys() + ["c(x)"],
    )

    mu_c, sd_c = mean[0, -1].item(), sigma[0, -1].item()
    p_feasible = 0.5 * float(torch.erfc(torch.tensor(mu_c / (sd_c * np.sqrt(2.0)))))
    print(f"P(c(x) <= 0) = P({outcome_constraint['objective']} "
          f"{outcome_constraint['direction']} {outcome_constraint['threshold']}) "
          f"= {p_feasible:.1%}")
    return table.round(4)


inspect_proposal(model, proposal, domain, OUTCOME_CONSTRAINT, col_info)

### Progress of the campaign

The scalar that summarises a constrained multi-objective campaign is the
hypervolume dominated by the **feasible** Pareto front, relative to the
reference point of §4. It increases whenever an experiment either improves a
trade-off already on the front or opens a new one, and it is the quantity
qLogNEHVI is trying to increase in expectation.

In [ ]:
def campaign_progress(domain, experiments, outcome_constraint):
    # Feasible Pareto front, and the hypervolume it dominates.
    key       = outcome_constraint["objective"]
    threshold = float(outcome_constraint["threshold"])
    feasible  = (experiments[key] >= threshold if outcome_constraint["direction"] == ">="
                 else experiments[key] <= threshold)

    objective_feats = domain.outputs.features
    ref_point = torch.tensor([
        -float(f.objective.upper_bound) if isinstance(f.objective, MinimizeObjective)
        else float(f.objective.lower_bound)
        for f in objective_feats
    ], **TKWARGS)

    if not feasible.any():
        return experiments.iloc[[]], 0.0

    obj = torch.tensor([
        [(-1.0 if isinstance(f.objective, MinimizeObjective) else 1.0) * float(row[f.key])
         for f in objective_feats]
        for _, row in experiments[feasible].iterrows()
    ], **TKWARGS)

    on_front = is_non_dominated(obj)
    hypervolume = Hypervolume(ref_point=ref_point).compute(obj[on_front])
    return experiments[feasible][on_front.cpu().numpy()], hypervolume


front, hypervolume = campaign_progress(domain, EXPERIMENTS, OUTCOME_CONSTRAINT)
print(f"hypervolume of the feasible Pareto front: {hypervolume:.3f}")
front

## 7. The closed loop

Everything above is one turn of the loop. A campaign is that turn repeated: the
surrogates are refitted on every experiment recorded so far, a new proposal is
computed, the chemist performs it and records the outcomes. `run_experiment`
below stands for the bench: it receives a set of conditions and returns the
measured objectives.

Two properties are worth stating explicitly in a methods section. The models
are refitted from scratch at each iteration, so a campaign is fully determined
by the set of experiments recorded and by the seed — the order in which they
were run does not enter. And nothing is discarded: infeasible experiments
constrain the third GP exactly as feasible ones constrain the first two.

In [ ]:
def run_campaign(domain, experiments, outcome_constraint, run_experiment,
                 n_iterations, seed=None):
    # Iterate propose -> perform -> record, returning the campaign so far.
    experiments = experiments.copy()
    for iteration in range(n_iterations):
        proposal = suggest_next_experiment(
            domain, experiments, outcome_constraint, q=1,
            seed=None if seed is None else seed + iteration,
        )
        conditions = proposal.iloc[0].to_dict()
        outcomes   = run_experiment(conditions)          # <- the bench
        experiments = pd.concat(
            [experiments, pd.DataFrame([{**conditions, **outcomes}])],
            ignore_index=True,
        )
        _, hypervolume = campaign_progress(domain, experiments, outcome_constraint)
        print(f"[{iteration + 1:>2}/{n_iterations}] "
              + ", ".join(f"{k}={v:.2f}" if isinstance(v, float) else f"{k}={v}"
                          for k, v in conditions.items())
              + f" -> {outcomes}, hypervolume {hypervolume:.3f}")
    return experiments


# In this campaign run_experiment was a chemist at the bench. Substituting a
# response surface here turns the same loop into an in-silico benchmark.
#
# EXPERIMENTS = run_campaign(
#     domain, EXPERIMENTS, OUTCOME_CONSTRAINT, run_experiment,
#     n_iterations=20, seed=SEED,
# )

## Notes

**Discrete parameters.** During the maximisation of the acquisition function a
discrete parameter is relaxed over its range and snapped back to the nearest
declared level on decoding. Enumerating the levels through
`fixed_features_list`, as is done for categorical parameters, is exact but
costs one maximisation per level; the relaxation is used where the levels are
evenly spaced and ordered, as `RES` is here.

**Reproducibility.** Both random generators involved carry their own state: the
sampler of §2 and the quasi-Monte-Carlo sampler of §5. Neither is reached by
`np.random.seed` or `torch.manual_seed`, so a campaign is replayable only if a
seed is passed explicitly to both. Where two candidates score closely, an
unseeded run flips the arg-max and two otherwise identical campaigns diverge
from that point on.

**Cost.** One iteration is dominated by fitting the GPs, which is cubic in the
number of experiments recorded. At the scale of a bench campaign — tens to a
few hundred experiments — a proposal takes seconds on a laptop CPU.

## References

1. Daulton, S.; Balandat, M.; Bakshy, E. *Parallel Bayesian Optimization of
   Multiple Noisy Objectives with Expected Hypervolume Improvement.*
   NeurIPS **2021**. — qNEHVI
2. Ament, S.; Daulton, S.; Eriksson, D.; Balandat, M.; Bakshy, E.
   *Unexpected Improvements to Expected Improvement for Bayesian Optimization.*
   NeurIPS **2023**. — the log formulation
3. Balandat, M. *et al.* *BoTorch: A Framework for Efficient Monte-Carlo
   Bayesian Optimization.* NeurIPS **2020**.
4. Durholt, J. P. *et al.* *BoFire: Bayesian Optimization Framework Intended
   for Real Experiments.* **2024**, arXiv:2408.05040.
5. Shields, B. J. *et al.* *Bayesian reaction optimization as a tool for
   chemical synthesis.* *Nature* **2021**, *590*, 89–96.